## Exp2: CGM + time_since_meal

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import CGMTimePredictor

MODEL_NAME = 'CGM+time_since_meal (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = None

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_cgm_time'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_val_stability_summary.json'

In [3]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 5)]


In [4]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device), tsm.to(device), has_meal.to(device)), target.to(device)

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model
    model = CGMTimePredictor(**{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_cgm_time' /f'gated_cgm_time_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — CGM+time_since_meal (gated) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: CGMTimePredictor  params: 1,026,261


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.58 | MAE 7.20 | gRMSE 11.49 | Zone A 96.89% | A+B 99.66%
 30 min | RMSE 16.67 | MAE 11.26 | gRMSE 18.65 | Zone A 90.31% | A+B 99.36%
 60 min | RMSE 24.08 | MAE 16.00 | gRMSE 28.24 | Zone A 82.32% | A+B 98.83%
 90 min | RMSE 28.45 | MAE 19.07 | gRMSE 34.28 | Zone A 77.66% | A+B 98.42%
120 min | RMSE 31.29 | MAE 21.31 | gRMSE 38.01 | Zone A 73.62% | A+B 98.03%

--------------------------------------------------------------------------------
Validation results — CGM+time_since_meal (gated) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: CGMTimePredictor  params: 1,026,261


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.65 | MAE 7.32 | gRMSE 11.74 | Zone A 97.09% | A+B 99.65%
 30 min | RMSE 16.73 | MAE 11.28 | gRMSE 18.83 | Zone A 90.48% | A+B 99.35%
 60 min | RMSE 24.10 | MAE 15.91 | gRMSE 28.21 | Zone A 82.47% | A+B 98.78%
 90 min | RMSE 28.39 | MAE 18.92 | gRMSE 34.04 | Zone A 77.82% | A+B 98.36%
120 min | RMSE 31.23 | MAE 21.12 | gRMSE 37.82 | Zone A 73.80% | A+B 98.00%

--------------------------------------------------------------------------------
Validation results — CGM+time_since_meal (gated) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: CGMTimePredictor  params: 1,026,261


...

 15 min | RMSE 10.63 | MAE 7.26 | gRMSE 11.72 | Zone A 97.20% | A+B 99.65%
 30 min | RMSE 16.77 | MAE 11.28 | gRMSE 19.02 | Zone A 90.42% | A+B 99.33%
 60 min | RMSE 24.16 | MAE 15.89 | gRMSE 28.63 | Zone A 82.52% | A+B 98.79%
 90 min | RMSE 28.49 | MAE 19.02 | gRMSE 34.55 | Zone A 78.04% | A+B 98.40%
120 min | RMSE 31.28 | MAE 21.32 | gRMSE 38.29 | Zone A 73.70% | A+B 98.02%

--------------------------------------------------------------------------------
Validation results — CGM+time_since_meal (gated) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: CGMTimePredictor  params: 1,026,261


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.58 | MAE 7.18 | gRMSE 11.42 | Zone A 97.08% | A+B 99.68%
 30 min | RMSE 16.75 | MAE 11.23 | gRMSE 18.69 | Zone A 90.37% | A+B 99.38%
 60 min | RMSE 24.05 | MAE 15.82 | gRMSE 28.17 | Zone A 82.43% | A+B 98.80%
 90 min | RMSE 28.39 | MAE 18.83 | gRMSE 34.14 | Zone A 77.91% | A+B 98.42%
120 min | RMSE 31.28 | MAE 21.18 | gRMSE 38.00 | Zone A 74.11% | A+B 98.03%

--------------------------------------------------------------------------------
Validation results — CGM+time_since_meal (gated) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: CGMTimePredictor  params: 1,026,261


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.57 | MAE 7.17 | gRMSE 11.55 | Zone A 96.95% | A+B 99.66%
 30 min | RMSE 16.69 | MAE 11.23 | gRMSE 18.74 | Zone A 90.38% | A+B 99.36%
 60 min | RMSE 24.06 | MAE 15.89 | gRMSE 28.27 | Zone A 82.39% | A+B 98.83%
 90 min | RMSE 28.39 | MAE 18.92 | gRMSE 34.22 | Zone A 77.64% | A+B 98.40%
120 min | RMSE 31.31 | MAE 21.22 | gRMSE 38.12 | Zone A 74.00% | A+B 97.98%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [8]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [9]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM+time_since_meal (gated) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.58 +/- 0.07 (2.09) mg/dL   MAE = 7.22 +/- 0.09 (1.41)   gRMSE = 11.52 +/- 0.14 (2.92)   Zone A = 97.05 +/- 0.15 (2.73)%   Zone A+B = 99.66 +/- 0.01 (0.65)%
 30-min:  RMSE = 16.68 +/- 0.07 (3.61) mg/dL   MAE = 11.25 +/- 0.08 (2.35)   gRMSE = 18.69 +/- 0.15 (5.26)   Zone A = 90.36 +/- 0.14 (4.94)%   Zone A+B = 99.37 +/- 0.02 (1.00)%
 60-min:  RMSE = 24.07 +/- 0.06 (7.04) mg/dL   MAE = 15.93 +/- 0.11 (4.50)   gRMSE = 28.23 +/- 0.16 (10.40)   Zone A = 82.34 +/- 0.26 (4.45)%   Zone A+B = 98.81 +/- 0.02 (1.27)%
 90-min:  RMSE = 28.40 +/- 0.05 (10.27) mg/dL   MAE = 18.97 +/- 0.11 (6.71)   gRMSE = 34.17 +/- 0.17 (15.06)   Zone A = 77.80 +/- 0.17 (7.00)%   Zone A+B = 98.41 +/- 0.02 (1.87)%
120-min:  RMSE = 31.25 +/- 0.06 (12.14) mg/dL   MAE = 21.24 +/- 0.12 (8.07)   gRMSE = 37.94 +/- 0.18 (17.83)   Zone A = 73.76 +/- 0.30 (8.22)%   Zone A+B = 98.03 +/- 0.03 (2.40)%


In [11]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_cgm_time/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_cgm_time/gated_cgm_time_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_cgm_time/gated_cgm_time_stability_summary.json
